In [1]:
!pip install pinecone
!pip install deepface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.4 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 25.0
    Uninstalling packaging-25.0:
      Successfully uninstalled packaging-25.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.3/128.3 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 88.8 MB/s eta 0:00:00


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
FOLDER_PATH = "/content/drive/MyDrive/VIT Convocation Images/S7-SCOPE-Other-Branches"
REFERENCE_IMAGE = "/content/drive/MyDrive/SCR-20250823-ngve.jpeg"

In [ ]:
from google.colab import userdata
PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')
INDEX_NAME = "convocation-faces"
NAMESPACE = "convocation-faces"

In [ ]:
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec

from deepface import DeepFace

import os
from pathlib import Path
import numpy as np
from tqdm import tqdm

In [ ]:
pc = Pinecone(api_key=PINECONE_API_KEY)

In [ ]:
MODEL_NAME = "VGG-Face"
DETECTOR = "retinaface"

In [ ]:
# Embedding dimension check
embeddings = DeepFace.represent(img_path=REFERENCE_IMAGE, model_name=MODEL_NAME, detector_backend=DETECTOR)

In [ ]:
embedding_dim = len(embeddings[0]['embedding'])
embedding_dim

In [ ]:
embeddings[0].keys()

In [ ]:
if not pc.has_index(INDEX_NAME):
  pc.create_index(
      name=INDEX_NAME,
      vector_type="dense",
      dimension=embedding_dim,
      metric="cosine",
      spec=ServerlessSpec(
          cloud="aws",
          region="us-east-1"
      )
  )

index = pc.Index(INDEX_NAME)

In [ ]:
def embed_and_store_images_in_pinecone(folder_path, batch_size=5, max_retries=2):
    image_paths = get_image_paths(folder_path)

    if not image_paths:
        print("No images found in the specified folder!")
        return

    print(f"Found {len(image_paths)} images to process")

    vectors = []
    total_faces = 0
    total_batches_uploaded = 0
    failed_images = []  # Track failed images for retry

    def process_image(image_path, retry_attempt=0):
        """Helper function to process a single image"""
        try:
            embeddings = DeepFace.represent(img_path=image_path, model_name=MODEL_NAME, detector_backend=DETECTOR)

            faces_in_image = len(embeddings)
            image_vectors = []

            for i, embedding in enumerate(embeddings):
                vector = {
                    "id": image_path + "[SEP]Face #" + str(i+1),
                    "values": embedding['embedding'],
                    "metadata": {
                        "facial_area": str(embedding['facial_area']),
                        "image_path": image_path,
                        "face_id": i+1,
                        "face_confidence": str(embedding['face_confidence'])
                    }
                }
                image_vectors.append(vector)

            return image_vectors, faces_in_image, None

        except Exception as e:
            error_msg = f"Attempt {retry_attempt + 1}: {str(e)}"
            return [], 0, error_msg

    # Main processing loop
    with tqdm(total=len(image_paths), desc="Processing Images", unit="img") as pbar:
        for image_path in image_paths:
            # Update progress bar with current image name
            pbar.set_postfix({'Current': os.path.basename(image_path)})

            image_vectors, faces_in_image, error = process_image(image_path)

            if error:
                # Image failed, add to retry list
                failed_images.append({'path': image_path, 'error': error})
                pbar.set_postfix({'Error': f"Failed: {os.path.basename(image_path)}"})
                print(f"\nError processing {image_path}: {error}")
            else:
                # Image processed successfully
                total_faces += faces_in_image
                vectors.extend(image_vectors)

                # Update progress bar description with face count
                pbar.set_description(f"Processing Images (Found {faces_in_image} faces)")

                # Upload batch if we have enough vectors
                if len(vectors) >= batch_size:
                    with tqdm(total=1, desc=f"Uploading Batch {total_batches_uploaded + 1}",
                            leave=False, position=1) as batch_pbar:
                        index.upsert(vectors=vectors, namespace=NAMESPACE)
                        batch_pbar.update(1)

                    total_batches_uploaded += 1
                    vectors = []

                    # Update main progress bar postfix with batch info
                    pbar.set_postfix({
                        'Current': os.path.basename(image_path),
                        'Batches': total_batches_uploaded,
                        'Total Faces': total_faces
                    })

            # Update progress bar
            pbar.update(1)

    # Retry failed images
    if failed_images and max_retries > 0:
        print(f"\n" + "="*50)
        print(f"RETRYING {len(failed_images)} FAILED IMAGES")
        print(f"="*50)

        retry_count = 1
        while failed_images and retry_count <= max_retries:
            print(f"\nRetry attempt {retry_count}/{max_retries}")
            current_failures = failed_images.copy()
            failed_images = []  # Reset failed list for this retry

            with tqdm(total=len(current_failures), desc=f"Retry {retry_count}", unit="img") as retry_pbar:
                for failed_item in current_failures:
                    image_path = failed_item['path']
                    retry_pbar.set_postfix({'Current': os.path.basename(image_path)})

                    image_vectors, faces_in_image, error = process_image(image_path, retry_count)

                    if error:
                        # Still failed, add back to failed list
                        failed_images.append({'path': image_path, 'error': error})
                        retry_pbar.set_postfix({'Still Failed': os.path.basename(image_path)})
                        print(f"\nRetry failed for {image_path}: {error}")
                    else:
                        # Success on retry!
                        total_faces += faces_in_image
                        vectors.extend(image_vectors)
                        retry_pbar.set_postfix({'Success': os.path.basename(image_path)})
                        print(f"\n✓ Successfully processed {image_path} on retry {retry_count}")

                        # Upload batch if we have enough vectors
                        if len(vectors) >= batch_size:
                            with tqdm(total=1, desc=f"Uploading Batch {total_batches_uploaded + 1}",
                                    leave=False, position=1) as batch_pbar:
                                index.upsert(vectors=vectors, namespace=NAMESPACE)
                                batch_pbar.update(1)

                            total_batches_uploaded += 1
                            vectors = []

                    retry_pbar.update(1)

            retry_count += 1

    # Upload remaining vectors
    if vectors:
        print(f"\nUploading final batch of {len(vectors)} vectors...")
        with tqdm(total=1, desc="Final Batch Upload") as final_pbar:
            index.upsert(vectors=vectors, namespace=NAMESPACE)
            final_pbar.upda

In [ ]:
embed_and_store_images_in_pinecone(FOLDER_PATH)

In [ ]:
def find_images_in_pinecone(image_path, top_k=3):
  ebs = DeepFace.represent(img_path=image_path, model_name=MODEL_NAME, detector_backend=DETECTOR)

  for i, emb in enumerate(ebs):
    matches = index.query(
        namespace=NAMESPACE,
        vector=emb['embedding'],
        top_k=top_k,
        include_metadata=True,
        include_values=False
    )

    return_val = {
        "match_data": matches.matches,
        "ref_data": {
            "facial_area": str(emb['facial_area'])
        }
    }

    if matches is not None:
      yield return_val

In [ ]:
def plot_face_match(match_data, reference_image_path):
    """
    Simple function to plot reference and match images side by side with bounding boxes

    Args:
        match_data (dict): The match data from Pinecone query
        reference_image_path (str): Path to the reference/query image
    """

    # Extract data
    ref_data = match_data.get('ref_data', {})
    matches = match_data.get('match_data', [])

    if not matches:
        print("No matches found")
        return

    # Get first match
    first_match = matches[0]
    match_image_path = first_match['metadata']['image_path']
    score = first_match['score']

    # Parse facial areas
    ref_face_area = ast.literal_eval(ref_data['facial_area'])
    match_face_area = ast.literal_eval(first_match['metadata']['facial_area'])

    # Create side by side plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

    # Plot reference image (fix orientation)
    ref_img = Image.open(reference_image_path)
    # Auto-rotate based on EXIF orientation data
    from PIL import ImageOps
    ref_img = ImageOps.exif_transpose(ref_img)

    ax1.imshow(ref_img)
    ax1.set_title('Reference Face', fontsize=14, fontweight='bold')

    # Draw reference bounding box
    x, y, w, h = ref_face_area['x'], ref_face_area['y'], ref_face_area['w'], ref_face_area['h']
    rect1 = patches.Rectangle((x, y), w, h, linewidth=3, edgecolor='red', facecolor='none')
    ax1.add_patch(rect1)
    ax1.text(x, y-10, 'Query', color='red', fontsize=12, fontweight='bold')
    ax1.axis('off')

    # Plot match image (fix orientation)
    match_img = Image.open(match_image_path)
    # Auto-rotate based on EXIF orientation data
    from PIL import ImageOps
    match_img = ImageOps.exif_transpose(match_img)

    ax2.imshow(match_img)
    ax2.set_title(f'Best Match (Score: {score:.4f})', fontsize=14, fontweight='bold')

    # Draw match bounding box
    x, y, w, h = match_face_area['x'], match_face_area['y'], match_face_area['w'], match_face_area['h']
    rect2 = patches.Rectangle((x, y), w, h, linewidth=3, edgecolor='lime', facecolor='none')
    ax2.add_patch(rect2)
    ax2.text(x, y-10, 'Match', color='lime', fontsize=12, fontweight='bold')
    ax2.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
REFERENCE_IMAGE = "/content/drive/MyDrive/WhatsApp Image 2025-08-23 at 14.32.36.jpeg"
for i in find_images_in_pinecone(REFERENCE_IMAGE, top_k=3):
  print(i)
  plot_face_match(i, REFERENCE_IMAGE)